# DeepSkin — EfficientNetB2 + CBAM (Local/Office Server Version)

**Architecture:** EfficientNetB2 (ImageNet pretrained) → CBAM Attention → GAP → BN → Dropout(0.5) → Dense(sigmoid)

**Training:**
- Phase 1: Head + CBAM only (backbone frozen) — BinaryCrossentropy + class_weight
- Phase 2: + block7 SE gates (conservative unfreeze) — BinaryCrossentropy + class_weight

> **Before running:** put your dataset in `../assets/` next to this notebook, with `benign/` and `malignant/` subfolders (directly, or one level down inside assets).


## 1. GPU Check

In [1]:
import os, sys

# Must run before importing tensorflow
venv_base = sys.prefix
py_version = f"python{sys.version_info.major}.{sys.version_info.minor}"
site_pkgs = os.path.join(venv_base, 'lib', py_version, 'site-packages')

nvidia_libs = [
    os.path.join(site_pkgs, 'nvidia', d, 'lib') for d in
    ['cuda_runtime', 'cublas', 'cufft', 'cudnn', 'curand',
     'cusolver', 'cusparse', 'nccl', 'nvjitlink', 'cuda_nvcc']
]
existing_ld = os.environ.get('LD_LIBRARY_PATH', '')
new_paths = [p for p in nvidia_libs if os.path.exists(p)]
os.environ['LD_LIBRARY_PATH'] = ':'.join(new_paths) + (f":{existing_ld}" if existing_ld else "")
print("LD_LIBRARY_PATH set. sys.executable:", sys.executable)

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TF version:', tf.__version__)
print('GPU devices:', gpus)

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# Mixed precision -- fine to enable now that TF 2.21 + aligned CUDA/ptxas
# versions correctly support sm_120 (confirmed by the earlier isolated matmul test)
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('Policy:', tf.keras.mixed_precision.global_policy())

LD_LIBRARY_PATH set. sys.executable: /home/higainai/project/venv/bin/python


I0000 00:00:1783599052.953110  143175 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783599053.280553  143175 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783599054.550279  143175 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TF version: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Policy: <DTypePolicy "mixed_float16">


W0000 00:00:1783599056.029804  143175 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


## 2. Dataset Discovery (local assets/ folder)


In [2]:
import os
import glob

# ── Local dataset discovery (assets/ folder next to this notebook) ──
# Expects a PRE-SPLIT, deduplicated dataset:
#   assets/<...>/train/benign, assets/<...>/train/malignant
#   assets/<...>/val/benign,   assets/<...>/val/malignant
# (produced by dedup_and_split.py — see split_manifest.csv for provenance)
DATA_ROOT = os.path.join(os.getcwd(), '../assets')
print(f'Looking for dataset under: {DATA_ROOT}')

train_benign_hits = glob.glob(os.path.join(DATA_ROOT, '**', 'train', 'benign'), recursive=True)
if not train_benign_hits:
    raise FileNotFoundError(
        f"Could not find a 'train/benign' folder under {DATA_ROOT}. "
        f"Expected a pre-split structure like assets/<name>/train/benign, "
        f"assets/<name>/train/malignant, assets/<name>/val/benign, assets/<name>/val/malignant. "
        f"Run dedup_and_split.py first if you haven't already."
    )
train_dir = os.path.dirname(train_benign_hits[0])
val_dir   = os.path.join(os.path.dirname(train_dir), 'val')
if not os.path.isdir(os.path.join(val_dir, 'benign')):
    raise FileNotFoundError(f"Found train_dir={train_dir} but no matching val_dir at {val_dir}")

print(f'train_dir: {train_dir}')
print(f'val_dir:   {val_dir}')

num_benign_train    = len(os.listdir(os.path.join(train_dir, 'benign')))
num_malignant_train = len(os.listdir(os.path.join(train_dir, 'malignant')))
num_benign_val       = len(os.listdir(os.path.join(val_dir, 'benign')))
num_malignant_val    = len(os.listdir(os.path.join(val_dir, 'malignant')))

total_train = num_benign_train + num_malignant_train
total_val   = num_benign_val + num_malignant_val

print(f'Train — Benign: {num_benign_train}  Malignant: {num_malignant_train}  '
      f'Ratio: {num_benign_train/num_malignant_train:.2f}:1')
print(f'Val   — Benign: {num_benign_val}  Malignant: {num_malignant_val}  '
      f'Ratio: {num_benign_val/num_malignant_val:.2f}:1')

# ── Output / checkpoint paths (local, next to the notebook) ──
PROJECT_DIR    = os.path.join(os.getcwd(), 'DeepSkin_Project_CBAM')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
MODEL_FILE     = os.path.join(CHECKPOINT_DIR, 'efficientnet_b2_cbam_best.keras')
METADATA_FILE  = os.path.join(CHECKPOINT_DIR, 'training_metadata.json')
HISTORY_FILE   = os.path.join(PROJECT_DIR,    'training_history.json')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PROJECT_DIR,    exist_ok=True)
print(f'\nCheckpoints: {CHECKPOINT_DIR}')


Looking for dataset under: /home/higainai/project/v4_gpu_optimized/../assets
train_dir: /home/higainai/project/v4_gpu_optimized/../assets/DeepSkin_Data_Processed/train
val_dir:   /home/higainai/project/v4_gpu_optimized/../assets/DeepSkin_Data_Processed/val
Train — Benign: 12793  Malignant: 7472  Ratio: 1.71:1
Val   — Benign: 3198  Malignant: 1868  Ratio: 1.71:1

Checkpoints: /home/higainai/project/v4_gpu_optimized/DeepSkin_Project_CBAM/checkpoints


## 3. (Optional) Resume From an External Checkpoint Backup

If you're resuming from a checkpoint you copied in from elsewhere (e.g. another
machine), drop the files into `EXTERNAL_CHECKPOINT_DIR` below and run this cell.
Otherwise it's a no-op — safe to just run and move on.


In [3]:
import shutil

# Point this at a folder of externally-saved checkpoints if you have one.
EXTERNAL_CHECKPOINT_DIR = os.path.join(os.getcwd(), 'external_checkpoints')
if os.path.exists(EXTERNAL_CHECKPOINT_DIR):
    for fname in os.listdir(EXTERNAL_CHECKPOINT_DIR):
        src = os.path.join(EXTERNAL_CHECKPOINT_DIR, fname)
        dst = os.path.join(CHECKPOINT_DIR, fname)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f'Copied: {fname}')
    print('Checkpoint copy done.')
else:
    print('No external checkpoint folder found — starting fresh (normal for first run).')


No external checkpoint folder found — starting fresh (normal for first run).


## 4. Imports

In [4]:
import tensorflow as tf
import numpy as np
import os, json, time, glob
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Kaggle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    recall_score, precision_score, fbeta_score,
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    accuracy_score, f1_score, roc_auc_score
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.metrics import Precision, Recall, AUC
from tensorflow.keras import layers

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

TF version: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 5. Session Timer

Kept mostly for logging/progress purposes. Since this is running on your own
server (no Kaggle 9-hour session limit), the ceiling below is set very high so
it will never cut off a real training run.


In [5]:
SESSION_START_TIME  = time.time()
# No Kaggle session limit here — this is just for progress logging.
MAX_SESSION_HOURS   = 1000
MAX_SESSION_SECONDS = MAX_SESSION_HOURS * 3600

def time_remaining():
    remaining = MAX_SESSION_SECONDS - (time.time() - SESSION_START_TIME)
    h = int(remaining // 3600)
    m = int((remaining % 3600) // 60)
    return f'{h}h {m}m remaining (informational only)'

def session_nearly_over():
    return (time.time() - SESSION_START_TIME) > MAX_SESSION_SECONDS

print('Session timer started (local run, no hard limit).')


Session timer started (local run, no hard limit).


## 6. Focal Loss

**alpha=0.35** matches the 2.13:1 class ratio (mathematically recommended = 0.32).  
Higher alpha values (0.75, 0.90) caused model collapse in previous runs.

In [6]:
ALPHA_VALUE  = 0.35   # matches 2.13:1 class ratio
GAMMA_PHASE1 = 1.0    # gentler for Phase 1 head training
GAMMA_PHASE2 = 2.0    # standard focal loss for Phase 2 fine-tuning

class BinaryFocalLoss(tf.keras.losses.Loss):
    def __init__(self, alpha=0.35, gamma=1.0, name='binary_focal_loss'):
        super().__init__(name=name)
        self.alpha = alpha
        self.gamma = gamma

    def call(self, y_true, y_pred):
        # Cast to float32 BEFORE computation — prevents float16 overflow
        y_pred  = tf.cast(y_pred, tf.float32)
        y_true  = tf.cast(y_true, tf.float32)
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce     = -(y_true * tf.math.log(y_pred) +
                    (1 - y_true) * tf.math.log(1 - y_pred))
        p_t     = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * self.alpha + (1 - y_true) * (1 - self.alpha)
        focal_w = alpha_t * tf.pow(1.0 - p_t, self.gamma)
        return tf.reduce_mean(focal_w * bce)

    def get_config(self):
        return {'alpha': self.alpha, 'gamma': self.gamma, 'name': self.name}

print(f'BinaryFocalLoss ready.')
print(f'  Phase 1: alpha={ALPHA_VALUE}, gamma={GAMMA_PHASE1}')
print(f'  Phase 2: alpha={ALPHA_VALUE}, gamma={GAMMA_PHASE2}')

BinaryFocalLoss ready.
  Phase 1: alpha=0.35, gamma=1.0
  Phase 2: alpha=0.35, gamma=2.0


## 7. CBAM Attention Module

```
Input Feature Map
      │
      ▼
Channel Attention  ← GlobalAvgPool + GlobalMaxPool → shared MLP → sigmoid → scale
      │
      ▼
Spatial Attention  ← AvgPool + MaxPool (channel-wise) → Conv7x7 → sigmoid → scale
      │
      ▼
Refined Feature Map  (lesion highlighted, background suppressed)
```

In [7]:
class ChannelAttention(layers.Layer):
    def __init__(self, reduction_ratio=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = reduction_ratio

    def build(self, input_shape):
        channels = input_shape[-1]
        reduced  = max(1, channels // self.reduction_ratio)
        self.dense1 = layers.Dense(reduced,   activation='relu', use_bias=False)
        self.dense2 = layers.Dense(channels,  activation=None,   use_bias=False)
        super().build(input_shape)

    def call(self, x):
        avg   = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        avg   = self.dense2(self.dense1(avg))
        mx    = tf.reduce_max(x,  axis=[1, 2], keepdims=True)
        mx    = self.dense2(self.dense1(mx))
        scale = tf.sigmoid(avg + mx)
        return x * scale

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction_ratio': self.reduction_ratio})
        return cfg


class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
        self.conv = layers.Conv2D(
            filters=1, kernel_size=kernel_size,
            padding='same', activation='sigmoid', use_bias=False
        )

    def call(self, x):
        avg      = tf.reduce_mean(x, axis=-1, keepdims=True)
        mx       = tf.reduce_max(x,  axis=-1, keepdims=True)
        combined = tf.concat([avg, mx], axis=-1)
        return x * self.conv(combined)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'kernel_size': self.kernel_size})
        return cfg


class CBAM(layers.Layer):
    def __init__(self, reduction_ratio=16, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = reduction_ratio
        self.kernel_size     = kernel_size
        self.channel_att     = ChannelAttention(reduction_ratio)
        self.spatial_att     = SpatialAttention(kernel_size)

    def call(self, x):
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction_ratio': self.reduction_ratio,
                    'kernel_size': self.kernel_size})
        return cfg


CUSTOM_OBJECTS = {
    'BinaryFocalLoss' : BinaryFocalLoss,
    'CBAM'            : CBAM,
    'ChannelAttention': ChannelAttention,
    'SpatialAttention': SpatialAttention,
}
print('CBAM defined: ChannelAttention + SpatialAttention + CBAM')

CBAM defined: ChannelAttention + SpatialAttention + CBAM


## 8. Build Model

In [8]:
def build_model_with_cbam():
    from tensorflow.keras.applications import EfficientNetB2
    from tensorflow.keras.layers import (
        GlobalAveragePooling2D, Dropout, Dense, BatchNormalization
    )
    from tensorflow.keras.models import Model

    # WORKAROUND: Force model construction on CPU
    # Avoids tf.cast GPU op that fails with CUDA_ERROR_INVALID_PTX on SM 12.0
    # TF automatically uses GPU for all training ops via XLA
    with tf.device('/CPU:0'):
        base = EfficientNetB2(
            weights='imagenet',
            include_top=False,
            input_shape=(260, 260, 3),
            name='efficientnetb2'
        )
        base.trainable = False

        x   = CBAM(name='cbam')(base.output)
        x   = GlobalAveragePooling2D(name='gap')(x)
        x   = BatchNormalization(name='bn_head')(x)
        x   = Dropout(0.5, name='dropout_head')(x)
        out = Dense(1, activation='sigmoid',
                    name='output', dtype='float32')(x)

        model = Model(inputs=base.input, outputs=out,
                      name='DeepSkin_CBAM')

    print('Model built on CPU successfully.')
    print(f'Total params: {model.count_params():,}')
    return model, base


_tmp, _ = build_model_with_cbam()
TOP_ACT_IDX = next(
    i for i, l in enumerate(_tmp.layers)
    if l.name == 'top_activation'
)
print(f'top_activation index: {TOP_ACT_IDX}')
del _tmp

W0000 00:00:1783599056.197132  143175 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1783599056.280903  143175 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15040 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5090, pci bus id: 0000:01:00.0, compute capability: 12.0a
E0000 00:00:1783599056.301426  143175 util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


Model built on CPU successfully.
Total params: 8,023,516
top_activation index: 339


## 9. Data Generators

In [9]:
# ── Optimized tf.data input pipeline ──
AUTOTUNE   = tf.data.AUTOTUNE
BATCH_SIZE = 256
IMG_SIZE   = (260, 260)

# class_names fixed and sorted — alphabetical: benign=0, malignant=1
class_names = sorted(os.listdir(train_dir))
print('Class names (index order):', class_names)

# WORKAROUND: Build augmentation layers on CPU
# Keras RandomFlip/RandomRotation etc. call tf.cast during __init__
# which triggers CUDA_ERROR_INVALID_PTX on SM 12.0 with TF 2.19/CUDA 12.5
# Forcing CPU context avoids the GPU cast op during layer construction
with tf.device('/CPU:0'):
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal_and_vertical'),
        tf.keras.layers.RandomRotation(30 / 360),
        tf.keras.layers.RandomTranslation(0.1, 0.1),
        tf.keras.layers.RandomZoom(0.15),
        tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255)),
    ], name='data_augmentation')
print('Augmentation pipeline built on CPU.')


def count_files(directory):
    """Cheap count via os.listdir -- avoids decoding the whole dataset just to
    get its length. FIX: the previous version used `sum(1 for _ in ds)`, which
    forces a full, unbatched, one-image-at-a-time decode+resize pass through
    the entire dataset just to count files -- this is what caused the 100+
    minute hang. Directory listing gets the same number in milliseconds, and
    we need this count anyway for class weights below."""
    total = 0
    for cls in class_names:
        cls_dir = os.path.join(directory, cls)
        total += len([f for f in os.listdir(cls_dir)
                       if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    return total


def make_dataset(directory, training, cache_path=None):
    n_samples = count_files(directory)   # fast: directory listing, not decoding

    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        labels='inferred',
        label_mode='binary',
        class_names=class_names,
        image_size=IMG_SIZE,
        batch_size=None,
        shuffle=training,
        seed=42,
    )

    # Cast to float32 — pixels stay [0,255] for EfficientNetB2
    ds = ds.map(
        lambda x, y: (tf.cast(x, tf.float32), tf.cast(y, tf.float32)),
        num_parallel_calls=AUTOTUNE
    )

    if cache_path:
        ds = ds.cache(cache_path)
    else:
        ds = ds.cache()

    if training:
        ds = ds.shuffle(
            buffer_size=n_samples,
            seed=42,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(BATCH_SIZE)

    if training:
        # FIX: clip after brightness augmentation prevents values outside [0,255]
        # Augmentation runs on CPU matching where layers were built
        ds = ds.map(
            lambda x, y: (
                tf.clip_by_value(
                    data_augmentation(x, training=True),
                    0.0, 255.0
                ),
                y
            ),
            num_parallel_calls=AUTOTUNE
        )

    # prefetch(2) — avoids AUTOTUNE requesting 200MB+ RAM
    ds = ds.prefetch(2)
    return ds, n_samples


train_dataset, n_train = make_dataset(train_dir, training=True, cache_path='/tmp/tf_cache_train')
val_dataset,   n_val   = make_dataset(val_dir,   training=False, cache_path='/tmp/tf_cache_val')


class DatasetMetadata:
    """Shim exposing .class_indices, .classes, __len__ for callbacks."""
    def __init__(self, dataset, directory, n_samples, batch_size, class_names):
        self.directory     = directory
        self.target_size   = IMG_SIZE
        self.n_samples     = n_samples
        self.batch_size    = batch_size
        self.class_indices = {name: i for i, name in enumerate(class_names)}
        self._classes      = None

    def __len__(self):
        return int(np.ceil(self.n_samples / self.batch_size))

    def reset(self):
        pass

    @property
    def classes(self):
        if self._classes is None:
            labels = []
            for class_name, idx in sorted(
                self.class_indices.items(), key=lambda x: x[1]
            ):
                class_dir = os.path.join(self.directory, class_name)
                if os.path.isdir(class_dir):
                    n_files = len([
                        f for f in os.listdir(class_dir)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
                    ])
                    labels.extend([idx] * n_files)
            self._classes = np.array(labels, dtype=np.int32)
            print(f'  [DatasetMetadata] {os.path.basename(self.directory)}: '
                  f'benign={int((self._classes==0).sum())}, '
                  f'malignant={int((self._classes==1).sum())}')
        return self._classes


train_generator      = DatasetMetadata(
    train_dataset, train_dir, n_train, BATCH_SIZE, class_names
)
validation_generator = DatasetMetadata(
    val_dataset, val_dir, n_val, BATCH_SIZE, class_names
)

# Class weights from train split only
num_b = len([f for f in os.listdir(os.path.join(train_dir, 'benign'))
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
num_m = len([f for f in os.listdir(os.path.join(train_dir, 'malignant'))
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
tot   = num_b + num_m
class_weight_dict = {0: tot / (2 * num_b), 1: tot / (2 * num_m)}

print(f'\nClass indices : {train_generator.class_indices}')
print(f'Train samples : {n_train}  ({len(train_generator)} batches of {BATCH_SIZE})')
print(f'Val samples   : {n_val}  ({len(validation_generator)} batches of {BATCH_SIZE})')
print(f'Class weights : {class_weight_dict}')

# Pixel range sanity check on CPU
sample_x, sample_y = next(iter(train_dataset))
print(f'\nSample batch pixel range: '
      f'[{sample_x.numpy().min():.1f}, {sample_x.numpy().max():.1f}]'
      f'  (should be [0.0, 255.0])')
print(f'Sample labels: {set(sample_y.numpy().flatten().tolist())}  '
      f'(should be {{0.0, 1.0}})')

Class names (index order): ['benign', 'malignant']
Augmentation pipeline built on CPU.
Found 20265 files belonging to 2 classes.
Found 5066 files belonging to 2 classes.

Class indices : {'benign': 0, 'malignant': 1}
Train samples : 20265  (80 batches of 256)
Val samples   : 5066  (20 batches of 256)
Class weights : {0: 0.7920347064801063, 1: 1.3560626338329764}


I0000 00:00:1783599068.353667  143314 shuffle_dataset_op.cc:453] ShuffleDatasetV3:9: Filling up shuffle buffer (this may take a while): 12207 of 20265
I0000 00:00:1783599070.451417  143314 shuffle_dataset_op.cc:483] Shuffle buffer filled.



Sample batch pixel range: [0.0, 255.0]  (should be [0.0, 255.0])
Sample labels: {0.0, 1.0}  (should be {0.0, 1.0})


## 10. Callbacks

In [10]:
class RealRecallCallback(tf.keras.callbacks.Callback):
    """Reports true recall/precision/F2 at multiple thresholds.
    Workaround for Keras Recall metric returning 0.0 with focal loss.
    Takes the raw tf.data.Dataset + a precomputed labels array (via the
    DatasetMetadata shim's .classes), since tf.data.Dataset has no .reset()."""
    def __init__(self, val_dataset, val_labels):
        super().__init__()
        self.val_dataset = val_dataset
        self.val_labels  = val_labels

    def on_epoch_end(self, epoch, logs=None):
        preds  = self.model.predict(self.val_dataset, verbose=0).flatten()
        labels = self.val_labels
        print(f'\n  [RealRecall] Epoch {epoch+1}:')
        for thresh in [0.3, 0.4, 0.5]:
            y_t = (preds >= thresh).astype(int)
            r   = recall_score(labels, y_t, zero_division=0)
            p   = precision_score(labels, y_t, zero_division=0)
            f2  = fbeta_score(labels, y_t, beta=2, zero_division=0)
            print(f'    thresh={thresh}: recall={r:.4f}  '
                  f'precision={p:.4f}  F2={f2:.4f}')


class FullModelCheckpoint(tf.keras.callbacks.Callback):
    """Saves best model (val_auc_pr) + latest epoch for reliable resume."""
    def __init__(self, model_file, meta_file, phase, current_epoch=0):
        super().__init__()
        self.model_file    = model_file
        self.meta_file     = meta_file
        self.phase         = phase
        self.current_epoch = current_epoch
        self.best_metric   = float('-inf')
        self.latest_file   = model_file.replace('.keras', '_latest.keras')
        self.latest_meta   = meta_file.replace('.json',  '_latest.json')

    def on_epoch_end(self, epoch, logs=None):
        self.current_epoch = epoch + 1
        metric = logs.get('val_auc_pr', float('-inf'))

        # Always save latest (for resume after timeout)
        try:
            self.model.save(self.latest_file)
            with open(self.latest_meta, 'w') as f:
                json.dump({'phase': self.phase, 'epoch': self.current_epoch,
                           'val_auc_pr': float(metric)}, f)
        except Exception as e:
            print(f'  Latest save failed: {e}')

        # Save best model
        if metric > self.best_metric:
            self.best_metric = metric
            try:
                self.model.save(self.model_file)
                with open(self.meta_file, 'w') as f:
                    json.dump({'phase': self.phase, 'epoch': self.current_epoch,
                               'val_auc_pr': float(metric)}, f)
                print(f'  ✓ Best saved — epoch {self.current_epoch}, '
                      f'val_auc_pr={metric:.4f}')
            except Exception as e:
                print(f'  Best save failed: {e}')
        else:
            print(f'  No improvement (best={self.best_metric:.4f})')


class SessionTimeoutCallback(tf.keras.callbacks.Callback):
    """Stops training safely before Kaggle session expires."""
    def on_epoch_end(self, epoch, logs=None):
        print(f'  {time_remaining()}')
        if session_nearly_over():
            print('WARNING: Session nearly over — stopping training safely.')
            self.model.stop_training = True


tracking_metrics = [
    'accuracy',
    Precision(name='precision', thresholds=0.5),
    Recall(name='recall',       thresholds=0.5),
    AUC(name='auc_roc', curve='ROC'),
    AUC(name='auc_pr',  curve='PR'),
]

base_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc_pr', patience=8, mode='max',
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc_pr', factor=0.5, patience=4,
        mode='max', verbose=1
    ),
]

real_recall_cb = RealRecallCallback(val_dataset, validation_generator.classes)
print('Callbacks ready.')


  [DatasetMetadata] val: benign=3198, malignant=1868
Callbacks ready.


## 11. Load Checkpoint or Start Fresh

In [11]:
# Prefer latest checkpoint (saved every epoch) over best (saved on improvement)
LATEST_MODEL = MODEL_FILE.replace('.keras', '_latest.keras')
LATEST_META  = METADATA_FILE.replace('.json',  '_latest.json')

RESUME_MODEL = LATEST_MODEL if os.path.exists(LATEST_MODEL) else MODEL_FILE
RESUME_META  = LATEST_META  if os.path.exists(LATEST_META)  else METADATA_FILE

if os.path.exists(RESUME_MODEL) and os.path.exists(RESUME_META):
    print('Checkpoint found — loading...')
    model = tf.keras.models.load_model(
        RESUME_MODEL, custom_objects=CUSTOM_OBJECTS, compile=False
    )
    with open(RESUME_META, 'r') as f:
        meta = json.load(f)
    start_phase = meta['phase']
    start_epoch = meta['epoch']
    print(f'Resumed: phase={start_phase}, epoch={start_epoch}, '
          f'val_auc_pr={meta["val_auc_pr"]:.4f}')
    if start_phase == 'complete':
        print('Training already complete. Run evaluation cells below.')
else:
    print('No checkpoint — starting fresh.')
    model, _ = build_model_with_cbam()
    start_phase = 'phase1'
    start_epoch = 0
    print(f'start_phase={start_phase}, start_epoch={start_epoch}')

Checkpoint found — loading...


/home/higainai/project/venv/lib/python3.12/site-packages/keras/src/layers/layer.py:431: UserWarning: `build()` was called on layer 'cbam', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Resumed: phase=phase2, epoch=11, val_auc_pr=0.8079


## 12. Phase 1 — Train Head + CBAM

Backbone frozen. BinaryCrossentropy + class_weight.  
Learning rate: 1e-3 (proposal spec).  
Augmentation: rotation, shift, zoom, flip, brightness via ImageDataGenerator.

In [12]:
if start_phase == 'phase1':
    print('=== Phase 1: Training Head + CBAM (backbone frozen) ===')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=tracking_metrics,
        jit_compile=False
    )

    trainable = sum([tf.size(w).numpy() for w in model.trainable_weights])
    print(f'Trainable params: {trainable:,}  (CBAM + head only)')

    phase1_ckpt = FullModelCheckpoint(
        MODEL_FILE, METADATA_FILE, 'phase1', start_epoch
    )

    history_phase1 = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=10,
        initial_epoch=start_epoch,
        class_weight=class_weight_dict,
        callbacks=base_callbacks + [
            phase1_ckpt, real_recall_cb, SessionTimeoutCallback()
        ]
    )

    # Save Phase 1 backup before transitioning
    phase1_backup = os.path.join(PROJECT_DIR, 'phase1_complete_model.keras')
    model.save(phase1_backup)
    print(f'Phase 1 backup: {phase1_backup}')

    with open(METADATA_FILE, 'w') as f:
        json.dump({'phase': 'phase2', 'epoch': 0, 'val_auc_pr': 0.0}, f)

    actual = len(history_phase1.history['loss'])
    print(f'Phase 1 complete. Ran {actual} epochs.')
    start_phase = 'phase2'
    start_epoch = 0

else:
    print(f'Skipping Phase 1 (start_phase={start_phase})')


Skipping Phase 1 (start_phase=phase2)


## 13. Phase 2 — Fine-tune Block7 SE Gates + CBAM

**Conservative unfreeze:** Only block7a/b SE squeeze-excite + dwconv layers (~786K params).  
Massive expand/project convs (743K each) stay frozen to prevent catastrophic forgetting.  
All BatchNorm layers frozen for stability.  
Safety check: raises error if trainable params exceed 900K.

In [ ]:
if start_phase == 'phase2':
    print('=== Phase 2: Fine-tuning Block7 SE Gates + CBAM ===')

    # Load Phase 1 backup cleanly to avoid corrupted weights
    phase1_backup = os.path.join(PROJECT_DIR, 'phase1_complete_model.keras')
    if os.path.exists(phase1_backup):
        model = tf.keras.models.load_model(
            phase1_backup, custom_objects=CUSTOM_OBJECTS, compile=False
        )
        print('Loaded Phase 1 backup cleanly.')
    else:
        print('Warning: No Phase 1 backup found — using current model.')

    # Step 1: Freeze ALL layers
    for layer in model.layers:
        layer.trainable = False

    # Step 2: Unfreeze ONLY conservative lightweight layers
    # block7 SE gates: small param layers that learn channel attention
    # Avoids unfreezing massive expand/project convs (743K params each)
    UNFREEZE_NAMES = {
        'block7b_dwconv',    # 19,008 params
        'block7b_se_reduce', # 185,944 params
        'block7b_se_expand', # 187,968 params
        'block7a_dwconv',    # 19,008 params
        'block7a_se_reduce', # 185,944 params
        'block7a_se_expand', # 187,968 params
        # Head (already trained Phase 1, continue refining)
        'cbam',
        'gap',
        'bn_head',
        'dropout_head',
        'output',
    }
    for layer in model.layers:
        if layer.name in UNFREEZE_NAMES:
            layer.trainable = True

    # Step 3: Keep ALL BatchNorm frozen for training stability
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    # Verify counts
    trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
    trainable_names  = [l.name for l in model.layers if l.trainable]
    bn_frozen        = sum(1 for l in model.layers
                           if isinstance(l, tf.keras.layers.BatchNormalization))

    print(f'Trainable params: {trainable_params:,}')
    print(f'BatchNorm frozen: {bn_frozen}')
    print(f'Trainable layers: {trainable_names}')

    # Safety check — stop before collapse
    if trainable_params > 900_000:
        raise ValueError(
            f'Too many trainable params ({trainable_params:,}). '
            f'Expected ~786,000. Check UNFREEZE_NAMES list.'
        )
    print(f'Param count OK ({trainable_params:,} < 900,000). Proceeding.')

    # Compile: binary_crossentropy safer than focal loss for fine-tuning
    model.compile(
        optimizer=tf.keras.mixed_precision.LossScaleOptimizer(
            tf.keras.optimizers.Adam(learning_rate=1e-5, clipnorm=1.0)
        ),
        loss='binary_crossentropy',
        metrics=tracking_metrics,
        jit_compile=False
    )

    phase2_ckpt = FullModelCheckpoint(
        MODEL_FILE, METADATA_FILE, 'phase2', start_epoch
    )

    PHASE1_EPOCHS    = 10
    PHASE2_MAX_EPOCH = 50  # total budget (absolute epoch count, not relative)

    history_phase2 = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=PHASE2_MAX_EPOCH,   # FIXED: absolute target epoch, not a relative count
                                   # (model.fit's `epochs` + `initial_epoch` together define
                                   # an absolute range; using a relative count here previously
                                   # caused a resumed run to see initial_epoch >= epochs and
                                   # silently train for zero epochs)
        initial_epoch=start_epoch,
        class_weight=class_weight_dict,
        callbacks=base_callbacks + [
            phase2_ckpt, real_recall_cb, SessionTimeoutCallback()
        ]
    )

    if len(history_phase2.history.get('loss', [])) == 0:
        print(f'No epochs were run (initial_epoch={start_epoch} >= epochs={PHASE2_MAX_EPOCH}).')
        print('Training budget already exhausted for this phase.')
        actual = 0
    else:
        actual = len(history_phase2.history['loss'])
        print(f'Phase 2 ran for {actual} epochs.')

    final_path = os.path.join(PROJECT_DIR, 'final_model_cbam.keras')
    model.save(final_path)
    print(f'Final model: {final_path}')

    with open(METADATA_FILE, 'w') as f:
        json.dump({
            'phase': 'complete', 'epoch': actual,
            'val_auc_pr': float(max(history_phase2.history.get('val_auc_pr', [0])))
        }, f)

    start_phase = 'complete'

else:
    print(f'Skipping Phase 2 (start_phase={start_phase})')


=== Phase 2: Fine-tuning Block7 SE Gates + CBAM ===
Loaded Phase 1 backup cleanly.
Trainable params: 784,559
BatchNorm frozen: 70
Trainable layers: ['block7a_dwconv', 'block7a_se_reduce', 'block7a_se_expand', 'block7b_dwconv', 'block7b_se_reduce', 'block7b_se_expand', 'cbam', 'gap', 'dropout_head', 'output']
Param count OK (784,559 < 900,000). Proceeding.
Epoch 12/50


/home/higainai/project/venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1783599092.291703  143372 shuffle_dataset_op.cc:453] ShuffleDatasetV3:9: Filling up shuffle buffer (this may take a while): 17505 of 20265
I0000 00:00:1783599094.425734  143372 shuffle_dataset_op.cc:483] Shuffle buffer filled.
I0000 00:00:1783599096.733641  143232 cuda_dnn.cc:461] Loaded cuDNN version 91002


80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step - accuracy: 0.7713 - auc_pr: 0.7764 - auc_roc: 0.8599 - loss: 0.4630 - precision: 0.6548 - recall: 0.8031  ✓ Best saved — epoch 12, val_auc_pr=0.8037

  [RealRecall] Epoch 12:
    thresh=0.3: recall=0.9224  precision=0.5688  F2=0.8204
    thresh=0.4: recall=0.8731  precision=0.6356  F2=0.8124
    thresh=0.5: recall=0.7960  precision=0.6978  F2=0.7742
  999h 58m remaining (informational only)
80/80 ━━━━━━━━━━━━━━━━━━━━ 57s 374ms/step - accuracy: 0.7713 - auc_pr: 0.7764 - auc_roc: 0.8599 - loss: 0.4630 - precision: 0.6548 - recall: 0.8031 - val_accuracy: 0.7977 - val_auc_pr: 0.8037 - val_auc_roc: 0.8773 - val_loss: 0.4353 - val_precision: 0.6978 - val_recall: 0.7960 - learning_rate: 1.0000e-05
Epoch 13/50


I0000 00:00:1783599144.198331  143780 shuffle_dataset_op.cc:453] ShuffleDatasetV3:9: Filling up shuffle buffer (this may take a while): 9050 of 20265
I0000 00:00:1783599150.871446  143780 shuffle_dataset_op.cc:483] Shuffle buffer filled.


79/80 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step - accuracy: 0.7745 - auc_pr: 0.7873 - auc_roc: 0.8651 - loss: 0.4548 - precision: 0.6615 - recall: 0.7960  ✓ Best saved — epoch 13, val_auc_pr=0.8039

  [RealRecall] Epoch 13:
    thresh=0.3: recall=0.9117  precision=0.5777  F2=0.8172
    thresh=0.4: recall=0.8651  precision=0.6441  F2=0.8095
    thresh=0.5: recall=0.7875  precision=0.7032  F2=0.7690
  999h 57m remaining (informational only)
80/80 ━━━━━━━━━━━━━━━━━━━━ 43s 304ms/step - accuracy: 0.7745 - auc_pr: 0.7873 - auc_roc: 0.8651 - loss: 0.4547 - precision: 0.6614 - recall: 0.7959 - val_accuracy: 0.7991 - val_auc_pr: 0.8039 - val_auc_roc: 0.8777 - val_loss: 0.4307 - val_precision: 0.7032 - val_recall: 0.7875 - learning_rate: 1.0000e-05
Epoch 14/50


...

79/80 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - accuracy: 0.7773 - auc_pr: 0.7870 - auc_roc: 0.8672 - loss: 0.4504 - precision: 0.6640 - recall: 0.8026  No improvement (best=0.8088)

  [RealRecall] Epoch 23:
    thresh=0.3: recall=0.9234  precision=0.5802  F2=0.8258
    thresh=0.4: recall=0.8758  precision=0.6421  F2=0.8164
    thresh=0.5: recall=0.8009  precision=0.6974  F2=0.7778
  999h 50m remaining (informational only)
80/80 ━━━━━━━━━━━━━━━━━━━━ 43s 270ms/step - accuracy: 0.7775 - auc_pr: 0.7872 - auc_roc: 0.8674 - loss: 0.4501 - precision: 0.6640 - recall: 0.8029 - val_accuracy: 0.7985 - val_auc_pr: 0.8088 - val_auc_roc: 0.8808 - val_loss: 0.4282 - val_precision: 0.6974 - val_recall: 0.8009 - learning_rate: 1.0000e-05
Epoch 24/50


I0000 00:00:1783599622.670266  144862 shuffle_dataset_op.cc:453] ShuffleDatasetV3:9: Filling up shuffle buffer (this may take a while): 9656 of 20265
I0000 00:00:1783599632.386814  144862 shuffle_dataset_op.cc:483] Shuffle buffer filled.


79/80 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step - accuracy: 0.7742 - auc_pr: 0.7903 - auc_roc: 0.8668 - loss: 0.4509 - precision: 0.6601 - recall: 0.7983  ✓ Best saved — epoch 24, val_auc_pr=0.8094

  [RealRecall] Epoch 24:
    thresh=0.3: recall=0.9192  precision=0.5856  F2=0.8252
    thresh=0.4: recall=0.8704  precision=0.6447  F2=0.8135
    thresh=0.5: recall=0.7918  precision=0.7016  F2=0.7719
  999h 50m remaining (informational only)
80/80 ━━━━━━━━━━━━━━━━━━━━ 43s 271ms/step - accuracy: 0.7741 - auc_pr: 0.7904 - auc_roc: 0.8667 - loss: 0.4511 - precision: 0.6603 - recall: 0.7980 - val_accuracy: 0.7991 - val_auc_pr: 0.8094 - val_auc_roc: 0.8810 - val_loss: 0.4260 - val_precision: 0.7016 - val_recall: 0.7918 - learning_rate: 1.0000e-05
Epoch 25/50


I0000 00:00:1783599665.991784  144981 shuffle_dataset_op.cc:453] ShuffleDatasetV3:9: Filling up shuffle buffer (this may take a while): 9854 of 20265
I0000 00:00:1783599675.520394  144981 shuffle_dataset_op.cc:483] Shuffle buffer filled.


40/80 ━━━━━━━━━━━━━━━━━━━━ 7s 198ms/step - accuracy: 0.7838 - auc_pr: 0.7895 - auc_roc: 0.8725 - loss: 0.4418 - precision: 0.6684 - recall: 0.8061

## 14. Save Training History

In [ ]:
HISTORY_KEYS = [
    'loss', 'val_loss', 'accuracy', 'val_accuracy',
    'recall', 'val_recall', 'precision', 'val_precision',
    'auc_roc', 'val_auc_roc', 'auc_pr', 'val_auc_pr'
]

try:
    hdata = {
        'phase1': {k: [float(v) for v in history_phase1.history.get(k, [])]
                   for k in HISTORY_KEYS},
        'phase2': {k: [float(v) for v in history_phase2.history.get(k, [])]
                   for k in HISTORY_KEYS},
    }
    with open(HISTORY_FILE, 'w') as f:
        json.dump(hdata, f)
    print(f'History saved: {HISTORY_FILE}')
except NameError:
    print('history_phase1/phase2 not in memory — skipping.')

## 15. Load Model & Generate Predictions

In [ ]:
EVAL_MODEL_PATH = os.path.join(PROJECT_DIR, 'final_model_cbam.keras')
if not os.path.exists(EVAL_MODEL_PATH):
    EVAL_MODEL_PATH = MODEL_FILE
    print(f'Final model not found, using best checkpoint.')

print(f'Loading: {EVAL_MODEL_PATH}')
eval_model = tf.keras.models.load_model(
    EVAL_MODEL_PATH, custom_objects=CUSTOM_OBJECTS, compile=False
)
print('Model loaded.')

# val_dataset is a tf.data.Dataset -- no .reset()/steps needed, it re-iterates
# cleanly every call. Labels come from the DatasetMetadata shim (validation_generator.classes).
y_pred_proba = eval_model.predict(val_dataset, verbose=1).flatten()
y_true = validation_generator.classes
y_pred = (y_pred_proba >= 0.5).astype(int)

print(f'Pred range: [{y_pred_proba.min():.4f}, {y_pred_proba.max():.4f}]')
print(f'Pred mean:  {y_pred_proba.mean():.4f}')
print(f'Predicted malignant (>=0.5): {y_pred.sum()}')
print(f'Actual malignant:            {y_true.sum()}')


## 16. Core Metrics

In [ ]:
accuracy    = accuracy_score(y_true, y_pred)
precision   = precision_score(y_true, y_pred, zero_division=0)
recall      = recall_score(y_true, y_pred, zero_division=0)
f1          = f1_score(y_true, y_pred, zero_division=0)
f2          = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
auc_roc_val = roc_auc_score(y_true, y_pred_proba)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)  # = recall
gmean       = np.sqrt(sensitivity * specificity)
fnr_val     = fn / (fn + tp)
fpr_val     = fp / (fp + tn)

all_metrics = [
    ('Accuracy',    accuracy,    0.90, '>'),
    ('Precision',   precision,   0.85, '>'),
    ('Recall',      recall,      0.90, '>'),
    ('Specificity', specificity, 0.85, '>'),
    ('F1-Score',    f1,          0.87, '>'),
    ('F2-Score',    f2,          0.85, '>'),
    ('AUC-ROC',     auc_roc_val, 0.95, '>'),
    ('G-Mean',      gmean,       0.85, '>'),
    ('FNR',         fnr_val,     0.10, '<'),
    ('FPR',         fpr_val,     0.15, '<'),
]

print('=' * 58)
print('     DeepSkin + CBAM  EVALUATION RESULTS')
print('=' * 58)
print(f'{"Metric":<20} {"Value":>10} {"Target":>10} {"Status":>8}')
print('-' * 58)
passed = 0
for name, val, tgt, direction in all_metrics:
    ok = val > tgt if direction == '>' else val < tgt
    if ok: passed += 1
    print(f'{name:<20} {val:>10.4f} {tgt:>10.2f} {"OK" if ok else "--":>8}')
print('=' * 58)
print(f'Targets met: {passed}/{len(all_metrics)}')
print(f'\nConfusion Matrix: TP={tp}  TN={tn}  FP={fp}  FN={fn}')
print('\nClassification Report:')
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, target_names=['Benign','Malignant']))

## 17. Threshold Analysis

In [ ]:
print('Threshold Sweep:')
print(f'{"Threshold":>10} {"Recall":>8} {"Precision":>10} {"F1":>8} {"F2":>8} {"Spec":>8}')
print('-' * 58)

best_f1_thresh = 0.5; best_f1 = 0.0
best_f2_thresh = 0.5; best_f2 = 0.0

for thresh in np.arange(0.1, 0.91, 0.05):
    y_t = (y_pred_proba >= thresh).astype(int)
    if y_t.sum() == 0: continue
    r   = recall_score(y_true, y_t, zero_division=0)
    p   = precision_score(y_true, y_t, zero_division=0)
    f1t = f1_score(y_true, y_t, zero_division=0)
    f2t = fbeta_score(y_true, y_t, beta=2, zero_division=0)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_true, y_t).ravel()
    sp  = tn_t / (tn_t + fp_t) if (tn_t + fp_t) > 0 else 0
    marks = []
    if f1t > best_f1: best_f1 = f1t; best_f1_thresh = thresh; marks.append('F1')
    if f2t > best_f2: best_f2 = f2t; best_f2_thresh = thresh; marks.append('F2')
    tag = f' <- best {" ".join(marks)}' if marks else ''
    print(f'{thresh:>10.2f} {r:>8.4f} {p:>10.4f} {f1t:>8.4f} {f2t:>8.4f} {sp:>8.4f}{tag}')

# Find threshold achieving recall >= 0.90
thresh90 = None
for t in np.linspace(0.01, 0.99, 1000):
    if recall_score(y_true, (y_pred_proba >= t).astype(int), zero_division=0) >= 0.90:
        thresh90 = t

print(f'\nBest F1 threshold : {best_f1_thresh:.2f}  (F1={best_f1:.4f})')
print(f'Best F2 threshold : {best_f2_thresh:.2f}  (F2={best_f2:.4f})')
if thresh90:
    r90 = recall_score(y_true, (y_pred_proba >= thresh90).astype(int), zero_division=0)
    p90 = precision_score(y_true, (y_pred_proba >= thresh90).astype(int), zero_division=0)
    print(f'Threshold for recall>=0.90: {thresh90:.4f} '
          f'(recall={r90:.4f}, precision={p90:.4f})')
else:
    print('Model cannot achieve recall>=0.90 at any threshold.')

## 18. Plots — Confusion Matrix, ROC, PR Curve

In [ ]:
y_pred_f2 = (y_pred_proba >= best_f2_thresh).astype(int)

# ── Confusion Matrices ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Confusion Matrices — DeepSkin + CBAM', fontsize=14, fontweight='bold')

for ax, y_p, title, cmap in [
    (axes[0], y_pred,    'Threshold = 0.50 (default)', 'Blues'),
    (axes[1], y_pred_f2, f'Threshold = {best_f2_thresh:.2f} (best F2)', 'Oranges'),
]:
    cm = confusion_matrix(y_true, y_p)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=['Benign','Malignant'],
                yticklabels=['Benign','Malignant'], ax=ax)
    ax.set_title(title); ax.set_ylabel('True'); ax.set_xlabel('Predicted')

plt.tight_layout()
cm_path = os.path.join(PROJECT_DIR, 'confusion_matrices.png')
plt.savefig(cm_path, dpi=150); plt.show()
print(f'Saved: {cm_path}')

In [ ]:
# ── ROC Curve ──
fpr_arr, tpr_arr, _ = roc_curve(y_true, y_pred_proba)
roc_auc_plot = auc(fpr_arr, tpr_arr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_arr, tpr_arr, 'darkorange', lw=2, label=f'AUC={roc_auc_plot:.4f}')
ax.plot([0,1],[0,1], 'navy', lw=1, linestyle='--', label='Random')
ax.scatter([fpr_val],[sensitivity], color='red', s=100, zorder=5,
           label=f'Default (recall={sensitivity:.3f})')
ax.axhline(y=0.90, color='green', linestyle=':', alpha=0.7, label='Target recall 0.90')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — DeepSkin + CBAM'); ax.legend(loc='lower right'); ax.grid(alpha=0.3)
roc_path = os.path.join(PROJECT_DIR, 'roc_curve.png')
plt.tight_layout(); plt.savefig(roc_path, dpi=150); plt.show()
print(f'AUC-ROC: {roc_auc_plot:.4f}  Saved: {roc_path}')

In [ ]:
# ── Precision-Recall Curve ──
prec_c, rec_c, _ = precision_recall_curve(y_true, y_pred_proba)
avg_prec = average_precision_score(y_true, y_pred_proba)
baseline = y_true.sum() / len(y_true)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(rec_c, prec_c, 'darkorange', lw=2, label=f'AP={avg_prec:.4f}')
ax.scatter([recall],[precision], color='red', s=100, zorder=5, label='Default thresh=0.50')
ax.scatter([recall_score(y_true, y_pred_f2)],
           [precision_score(y_true, y_pred_f2)],
           color='green', s=100, zorder=5, label=f'Best F2 thresh={best_f2_thresh:.2f}')
ax.axvline(x=0.90, color='purple', linestyle='--', alpha=0.7, label='Recall target')
ax.axhline(y=baseline, color='navy', linestyle='--', alpha=0.5,
           label=f'Baseline ({baseline:.2f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('PR Curve — DeepSkin + CBAM'); ax.legend(); ax.grid(alpha=0.3)
pr_path = os.path.join(PROJECT_DIR, 'pr_curve.png')
plt.tight_layout(); plt.savefig(pr_path, dpi=150); plt.show()
print(f'AP: {avg_prec:.4f}  Saved: {pr_path}')

## 19. Training Curves

In [ ]:
# Load history from file if not in memory
HIST_OK = False
try:
    _ = history_phase1; _ = history_phase2; HIST_OK = True
except NameError:
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, 'r') as f:
            hd = json.load(f)
        class H:
            def __init__(self, d): self.history = d
        history_phase1 = H(hd['phase1'])
        history_phase2 = H(hd['phase2'])
        HIST_OK = True
        print('History loaded from file.')
    else:
        print('No history file. Skipping training curves.')

if HIST_OK:
    p1 = history_phase1.history
    p2 = history_phase2.history
    ep1 = len(p1.get('loss', []))
    ep2 = len(p2.get('loss', []))
    x1  = range(1, ep1+1)
    x2  = range(ep1+1, ep1+ep2+1)

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('DeepSkin + CBAM Training History', fontsize=16, fontweight='bold')

    specs = [
        (axes[0,0], 'loss',     'Loss Curve',     None,  None),
        (axes[0,1], 'accuracy', 'Accuracy Curve', 0.90,  'Target 0.90'),
        (axes[1,0], 'recall',   'Recall Curve',   0.90,  'Target 0.90'),
        (axes[1,1], 'auc_pr',   'AUC-PR Curve',   0.95,  'Target 0.95'),
    ]

    for ax, key, title, tgt, tlbl in specs:
        tr1 = p1.get(key, []); vl1 = p1.get(f'val_{key}', [])
        tr2 = p2.get(key, []); vl2 = p2.get(f'val_{key}', [])
        if tr1: ax.plot(x1, tr1, 'b-o', ms=3, label='P1 Train')
        if vl1: ax.plot(x1, vl1, 'r-o', ms=3, label='P1 Val')
        if tr2: ax.plot(x2, tr2, 'b--o', ms=3, label='P2 Train')
        if vl2: ax.plot(x2, vl2, 'r--o', ms=3, label='P2 Val')
        if ep1 > 0:
            ax.axvline(x=ep1, color='gray', linestyle=':', alpha=0.7, label='P1→P2')
        if tgt is not None:
            ax.axhline(y=tgt, color='green', linestyle=':', alpha=0.7, label=tlbl)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    tc_path = os.path.join(PROJECT_DIR, 'training_curves.png')
    plt.savefig(tc_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {tc_path}')

## 20. CBAM Attention Map Visualisation

In [ ]:
import cv2

# Build attention map sub-model using CBAM layer directly
# Avoids get_layer('efficientnetb2') which crashes on flat saved model
cbam_layer = eval_model.get_layer('cbam')
cbam_input_tensor = cbam_layer.input

# --- FIXED: raw tf.* ops must be wrapped in a Lambda layer for Keras 3 / TF 2.16+ ---
ch_out = cbam_layer.channel_att(cbam_input_tensor)

combined = layers.Lambda(
    lambda t: tf.concat(
        [tf.reduce_mean(t, axis=-1, keepdims=True),
         tf.reduce_max(t,  axis=-1, keepdims=True)],
        axis=-1
    ),
    name='avg_max_concat'
)(ch_out)

spatial_mask = cbam_layer.spatial_att.conv(combined)

att_model = tf.keras.Model(
    inputs=eval_model.input,
    outputs=spatial_mask,
    name='attention_map_model'
)
print('Attention map model ready.')


def show_attention_maps(dataset, n=6):
    for batch_imgs, batch_labels in dataset.take(1):
        imgs   = batch_imgs.numpy()[:n]
        labels = batch_labels.numpy()[:n]
        break
    maps   = att_model.predict(imgs, verbose=0)
    preds  = eval_model.predict(imgs, verbose=0).flatten()

    fig, axes = plt.subplots(2, n, figsize=(3*n, 6))
    fig.suptitle('CBAM Spatial Attention — DeepSkin', fontsize=13, fontweight='bold')

    for i in range(n):
        img_u8 = np.clip(imgs[i], 0, 255).astype(np.uint8)
        t_lbl  = 'Malignant' if labels[i] == 1 else 'Benign'
        p_lbl  = 'Mal' if preds[i] >= 0.5 else 'Ben'
        axes[0, i].imshow(img_u8)
        axes[0, i].set_title(f'True:{t_lbl}\nPred:{p_lbl}({preds[i]:.2f})', fontsize=7)
        axes[0, i].axis('off')

        mask = maps[i, :, :, 0]
        mask = cv2.resize(mask, (260, 260))
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
        hm   = cv2.applyColorMap((mask*255).astype(np.uint8), cv2.COLORMAP_JET)
        hm   = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB)
        ov   = cv2.addWeighted(img_u8, 0.6, hm, 0.4, 0)
        axes[1, i].imshow(ov)
        axes[1, i].set_title('Attention', fontsize=7)
        axes[1, i].axis('off')

    plt.tight_layout()
    att_path = os.path.join(PROJECT_DIR, 'attention_maps.png')
    plt.savefig(att_path, dpi=150); plt.show()
    print(f'Saved: {att_path}')


show_attention_maps(val_dataset, n=6)

## 21. Final Summary

In [ ]:
avg_prec_summary = average_precision_score(y_true, y_pred_proba)

total_samples = total_train + total_val
num_benign    = num_benign_train + num_benign_val
num_malignant = num_malignant_train + num_malignant_val

print('=' * 58)
print('     DeepSkin + CBAM  FINAL SUMMARY')
print('=' * 58)
print(f'Model            : EfficientNetB2 + CBAM')
print(f'Dataset          : {total_samples:,} images  ({num_benign:,} benign / {num_malignant:,} malignant)')
print(f'')
print(f'--- At threshold = 0.50 ---')
print(f'Recall           : {recall:.4f}  (target >0.90)')
print(f'Precision        : {precision:.4f}  (target >0.85)')
print(f'F1-Score         : {f1:.4f}  (target >0.87)')
print(f'F2-Score         : {f2:.4f}')
print(f'AUC-ROC          : {auc_roc_val:.4f}  (target >0.95)')
print(f'AUC-PR           : {avg_prec_summary:.4f}')
print(f'G-Mean           : {gmean:.4f}')
print(f'FNR              : {fnr_val:.4f}  (target <0.10)')
print(f'')
print(f'--- Best thresholds ---')
print(f'Best F1 thresh   : {best_f1_thresh:.2f}  (F1={best_f1:.4f})')
print(f'Best F2 thresh   : {best_f2_thresh:.2f}  (F2={best_f2:.4f})')
if thresh90:
    print(f'Recall>=0.90 thresh: {thresh90:.4f}')
print(f'')
print(f'Targets met      : {passed}/{len(all_metrics)}')
print(f'TP={tp}  TN={tn}  FP={fp}  FN={fn}')
print('=' * 58)
print(f'Outputs in: {PROJECT_DIR}')

## 22. List Output Files

After training, download files from the **Output** tab on the right sidebar.

In [ ]:
print(f'Output files in {PROJECT_DIR}:')
for root, dirs, files in os.walk(PROJECT_DIR):
    level = root.replace(PROJECT_DIR, '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root)
    if level > 0:
        print(f'{indent}{folder}/')
    for file in files:
        size_mb = os.path.getsize(os.path.join(root, file)) / 1e6
        print(f'{indent}  {file}  ({size_mb:.1f} MB)')

## 23. [Improvement] Day 1 — Test-Time Augmentation (TTA)

Runs each validation image through the model multiple times (original + horizontal flip + vertical flip + small zoom crops) and averages the sigmoid outputs. No retraining required — this only changes inference. Targets the domain-shift finding specifically, since TTA tends to stabilize predictions for borderline/uncertain cases near the decision threshold.


In [ ]:
import tensorflow as tf
import numpy as np

def tta_predict(model, directory, class_names, img_size, n_tta=4, batch_size=BATCH_SIZE):
    """
    Test-Time Augmentation: averages predictions over the original image plus
    horizontal flip, vertical flip, and a zoom-in crop. Rebuilds a fresh,
    non-shuffling tf.data pipeline from the given directory.
    """
    base_ds = tf.keras.utils.image_dataset_from_directory(
        directory, labels='inferred', label_mode='binary',
        class_names=class_names, image_size=img_size,
        batch_size=batch_size, shuffle=False
    ).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=tf.data.AUTOTUNE) \
     .prefetch(tf.data.AUTOTUNE)

    y_true_tta = np.concatenate([y.numpy() for _, y in base_ds]).flatten().astype(int)

    augment_fns = [
        lambda x: x,                                     # original
        lambda x: tf.image.flip_left_right(x),            # horizontal flip
        lambda x: tf.image.flip_up_down(x),                # vertical flip
        lambda x: tf.image.central_crop(x, 0.85),          # mild zoom-in (needs resize back)
    ][:n_tta]

    all_preds = []
    for aug_fn in augment_fns:
        preds = []
        for batch_x, _ in base_ds:
            batch_aug = aug_fn(batch_x)
            if batch_aug.shape[1] != img_size[0] or batch_aug.shape[2] != img_size[1]:
                batch_aug = tf.image.resize(batch_aug, img_size)
            p = model.predict(batch_aug, verbose=0).flatten()
            preds.append(p)
        all_preds.append(np.concatenate(preds))

    y_pred_proba_tta = np.mean(all_preds, axis=0)
    return y_true_tta, y_pred_proba_tta


print('Running TTA on validation set (this takes ~n_tta times longer than a single pass)...')
y_true_tta, y_pred_proba_tta = tta_predict(
    eval_model, val_dir, class_names, IMG_SIZE, n_tta=4
)

# Compare against the non-TTA baseline already computed in Section 15/16
for thresh in [0.3, 0.4, 0.5]:
    y_p  = (y_pred_proba >= thresh).astype(int)
    y_pt = (y_pred_proba_tta >= thresh).astype(int)
    print(f"\nThreshold {thresh}:")
    print(f"  No TTA : recall={recall_score(y_true, y_p, zero_division=0):.4f}  "
          f"precision={precision_score(y_true, y_p, zero_division=0):.4f}  "
          f"F2={fbeta_score(y_true, y_p, beta=2, zero_division=0):.4f}")
    print(f"  +TTA   : recall={recall_score(y_true_tta, y_pt, zero_division=0):.4f}  "
          f"precision={precision_score(y_true_tta, y_pt, zero_division=0):.4f}  "
          f"F2={fbeta_score(y_true_tta, y_pt, beta=2, zero_division=0):.4f}")


## 24. [Improvement] Day 1 — Fine-Grained Threshold Sweep (Clinical Target)

The existing Section 17 sweep steps by 0.05 and optimizes F1/F2. This one steps by 0.02 and lets you pick a threshold by a stated clinical target (e.g. \"recall >= 0.90\") rather than by F-score alone, using the TTA predictions from Section 23.


In [ ]:

TARGET_RECALL = 0.90   # <-- set your clinical minimum-acceptable recall here

print(f"{'Threshold':>10} {'Recall':>8} {'Precision':>10} {'F2':>8}")
print('-' * 42)
candidates = []
for thresh in np.arange(0.05, 0.61, 0.02):
    y_t = (y_pred_proba_tta >= thresh).astype(int)
    if y_t.sum() == 0:
        continue
    r = recall_score(y_true_tta, y_t, zero_division=0)
    p = precision_score(y_true_tta, y_t, zero_division=0)
    f2t = fbeta_score(y_true_tta, y_t, beta=2, zero_division=0)
    print(f"{thresh:>10.2f} {r:>8.4f} {p:>10.4f} {f2t:>8.4f}")
    if r >= TARGET_RECALL:
        candidates.append((thresh, r, p, f2t))

if candidates:
    # Highest threshold that still meets the recall target -> best precision at that target
    chosen = max(candidates, key=lambda c: c[0])
    OPERATING_THRESHOLD = chosen[0]
    print(f"\nChosen operating threshold = {OPERATING_THRESHOLD:.2f} "
          f"(recall={chosen[1]:.4f}, precision={chosen[2]:.4f}) "
          f"-- highest threshold that still meets target recall >= {TARGET_RECALL}")
else:
    OPERATING_THRESHOLD = 0.3
    print(f"\nNo threshold met target recall >= {TARGET_RECALL}; "
          f"defaulting OPERATING_THRESHOLD={OPERATING_THRESHOLD}. Consider Day 3-4 retrain experiments.")


## 25. [Improvement] Day 2 — Confidence Calibration (Temperature Scaling)

Fits a single scalar temperature T on the validation logits to make the sigmoid output a more reliable probability estimate (a calibrated 0.6 should mean ~60% of such predictions are truly positive). Also defines an 'uncertain' rejection band around the operating threshold for flagging low-confidence predictions for manual review.


In [ ]:

from scipy.optimize import minimize_scalar
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def logit(p, eps=1e-7):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def nll_temperature(T, probs, labels):
    logits = logit(probs) / T
    p_scaled = sigmoid(logits)
    p_scaled = np.clip(p_scaled, 1e-7, 1 - 1e-7)
    return -np.mean(labels * np.log(p_scaled) + (1 - labels) * np.log(1 - p_scaled))

result = minimize_scalar(
    nll_temperature, bounds=(0.05, 5.0), method='bounded',
    args=(y_pred_proba_tta, y_true_tta)
)
TEMPERATURE = result.x
print(f"Fitted temperature: {TEMPERATURE:.4f}")

y_pred_proba_calibrated = sigmoid(logit(y_pred_proba_tta) / TEMPERATURE)

# Expected Calibration Error (ECE), before vs after
def expected_calibration_error(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            continue
        bin_acc = labels[mask].mean()
        bin_conf = probs[mask].mean()
        ece += (mask.sum() / len(probs)) * abs(bin_acc - bin_conf)
    return ece

ece_before = expected_calibration_error(y_pred_proba_tta, y_true_tta)
ece_after  = expected_calibration_error(y_pred_proba_calibrated, y_true_tta)
print(f"ECE before calibration: {ece_before:.4f}")
print(f"ECE after calibration:  {ece_after:.4f}")

# ── Uncertain / rejection band around the chosen operating threshold ──
BAND_WIDTH = 0.10   # +/- around OPERATING_THRESHOLD counted as "uncertain"
lower = max(0.0, OPERATING_THRESHOLD - BAND_WIDTH)
upper = min(1.0, OPERATING_THRESHOLD + BAND_WIDTH)

def classify_with_rejection(probs, low=lower, high=upper):
    labels = np.where(probs < low, 'benign',
              np.where(probs > high, 'malignant', 'uncertain_refer'))
    return labels

decisions = classify_with_rejection(y_pred_proba_calibrated)
n_uncertain = np.sum(decisions == 'uncertain_refer')
print(f"\nUncertain band: [{lower:.2f}, {upper:.2f}]")
print(f"Flagged as 'uncertain -> refer for review': {n_uncertain} / {len(decisions)} "
      f"({n_uncertain/len(decisions)*100:.1f}%)")

# Accuracy on the confidently-decided subset only
confident_mask = decisions != 'uncertain_refer'
if confident_mask.sum() > 0:
    confident_pred = (y_pred_proba_calibrated[confident_mask] >= OPERATING_THRESHOLD).astype(int)
    print(f"Recall on confident-only subset: "
          f"{recall_score(y_true_tta[confident_mask], confident_pred, zero_division=0):.4f}")
    print(f"Precision on confident-only subset: "
          f"{precision_score(y_true_tta[confident_mask], confident_pred, zero_division=0):.4f}")


## 26. [Improvement] Day 2 — Formalized Checkpoint Averaging

Formalizes the epoch-40 + epoch-50 combination as probability averaging across two saved checkpoints from Phase 2. Requires both checkpoints to exist on disk — if you only kept the 'best' and 'latest' checkpoints, this compares those two instead (documented explicitly either way).


In [ ]:

# Point these at whichever two Phase-2 checkpoints you saved.
# Defaults to best vs latest, since those are guaranteed to exist under the current checkpointing scheme.
CKPT_A_PATH = os.path.join(CHECKPOINT_DIR, 'efficientnet_b2_cbam_best.keras')
CKPT_B_PATH = os.path.join(CHECKPOINT_DIR, 'efficientnet_b2_cbam_best_latest.keras')

model_a = tf.keras.models.load_model(CKPT_A_PATH, custom_objects=CUSTOM_OBJECTS, compile=False)
model_b = tf.keras.models.load_model(CKPT_B_PATH, custom_objects=CUSTOM_OBJECTS, compile=False)

# val_dataset re-iterates cleanly on each predict() call -- no .reset() needed
proba_a = model_a.predict(val_dataset, verbose=0).flatten()
proba_b = model_b.predict(val_dataset, verbose=0).flatten()

# Probability averaging (simpler and more common than logit averaging for a 2-snapshot ensemble)
proba_ensemble = (proba_a + proba_b) / 2.0

for name, proba in [('Checkpoint A only', proba_a), ('Checkpoint B only', proba_b),
                     ('Ensemble (avg probability)', proba_ensemble)]:
    y_p = (proba >= OPERATING_THRESHOLD).astype(int)
    print(f"{name:28s}  recall={recall_score(y_true, y_p, zero_division=0):.4f}  "
          f"precision={precision_score(y_true, y_p, zero_division=0):.4f}  "
          f"F2={fbeta_score(y_true, y_p, beta=2, zero_division=0):.4f}  "
          f"AUC-PR={average_precision_score(y_true, proba):.4f}")


## 27. [Improvement] Day 3–4 — Phase 2 Variant Experiments

Resumes from the Phase 1 checkpoint (`phase1_complete_model.keras`) with two variables you can toggle: loss function (BCE vs Focal Loss) and unfreeze scope (block7 only vs block6+block7). Each variant saves to its own subfolder under `PROJECT_DIR/variants/` so it never overwrites your existing baseline checkpoints. Run this cell once per variant (change the config, rerun).


In [ ]:

# ── Variant configuration -- change these two lines per experiment run ──
VARIANT_NAME   = "focal_block7"     # e.g. "bce_block7" (baseline), "focal_block7", "focal_block6_7"
USE_FOCAL_LOSS = True                # False = BinaryCrossentropy (the original baseline)
EXTRA_UNFREEZE_BLOCK6 = False         # True = also unfreeze block6 SE gates (larger capacity bump)

VARIANT_DIR = os.path.join(PROJECT_DIR, 'variants', VARIANT_NAME)
os.makedirs(VARIANT_DIR, exist_ok=True)
VARIANT_MODEL_FILE    = os.path.join(VARIANT_DIR, 'model_best.keras')
VARIANT_METADATA_FILE = os.path.join(VARIANT_DIR, 'metadata.json')

phase1_backup = os.path.join(PROJECT_DIR, 'phase1_complete_model.keras')
model = tf.keras.models.load_model(phase1_backup, custom_objects=CUSTOM_OBJECTS, compile=False)
print(f"Loaded Phase 1 backup for variant: {VARIANT_NAME}")

for layer in model.layers:
    layer.trainable = False

UNFREEZE_NAMES = {
    'block7b_dwconv', 'block7b_se_reduce', 'block7b_se_expand',
    'block7a_dwconv', 'block7a_se_reduce', 'block7a_se_expand',
    'cbam', 'gap', 'bn_head', 'dropout_head', 'output',
}
if EXTRA_UNFREEZE_BLOCK6:
    UNFREEZE_NAMES |= {
        'block6a_dwconv', 'block6a_se_reduce', 'block6a_se_expand',
        'block6b_dwconv', 'block6b_se_reduce', 'block6b_se_expand',
        'block6c_dwconv', 'block6c_se_reduce', 'block6c_se_expand',
        'block6d_dwconv', 'block6d_se_reduce', 'block6d_se_expand',
    }
    # NOTE: verify these exact layer names exist in your EfficientNetB2 graph
    # (model.summary() or [l.name for l in model.layers if "block6" in l.name])
    # before relying on this -- EfficientNetB2 has more block6 sub-blocks than block7.

for layer in model.layers:
    if layer.name in UNFREEZE_NAMES:
        layer.trainable = True
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Trainable params: {trainable_params:,}")
if trainable_params > 2_000_000:
    raise ValueError(f"Trainable param count ({trainable_params:,}) looks too high -- check UNFREEZE_NAMES.")

loss_fn = BinaryFocalLoss(alpha=0.35, gamma=2.0) if USE_FOCAL_LOSS else 'binary_crossentropy'
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss=loss_fn, metrics=tracking_metrics,
              jit_compile=True)   # XLA: fuses ops, reduces memory-transfer overhead

variant_ckpt = FullModelCheckpoint(VARIANT_MODEL_FILE, VARIANT_METADATA_FILE, 'phase2_variant', 0)

history_variant = model.fit(
    train_dataset, validation_data=val_dataset,
    epochs=40, initial_epoch=0,
    class_weight=None if USE_FOCAL_LOSS else class_weight_dict,   # CHANGED: avoid double-correcting imbalance when focal loss's alpha already does it
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_auc_pr', patience=8, mode='max', restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_auc_pr', factor=0.5, patience=4, mode='max'),
        variant_ckpt, real_recall_cb,
    ]
)

model.save(os.path.join(VARIANT_DIR, 'model_final.keras'))
best_val_auc_pr = float(max(history_variant.history.get('val_auc_pr', [0])))
print(f"\nVariant '{VARIANT_NAME}' complete. Best val_auc_pr = {best_val_auc_pr:.4f}")
print(f"Compare this against your baseline val_auc_pr = 0.8192 to see if this variant helps.")


## 28. [Improvement] Day 5 — Multi-Source Validation & Augmentation Tuning (Manual Steps)

These two items need new data / a config rerun rather than a pure code addition here:

**Multi-source validation (recommended, addresses the domain-shift finding directly):**
1. Download an independent ISIC batch that you can *confirm* has zero overlap with training (run `overlap_check.py` against it first).
2. Hold this out as a **third, separate test set** — distinct from your current `val/` split — so you get a genuine, never-touched generalization number, not just another slice of the same combined pool.
3. Re-run Section 15-18's evaluation cells against this set using `flow_from_directory` pointed at the new folder in place of `validation_generator`.

**Augmentation tuning for domain robustness:**
1. In the Section 9-ish data-generator cell, add stronger color/contrast jitter to `train_datagen` -- e.g. `channel_shift_range=20.0` and a wider `brightness_range=[0.7, 1.3]` -- to simulate different camera/lighting pipelines.
2. Re-run a Phase 2 variant (Section 27) with this changed generator, and compare its performance specifically on the independent multi-source test set from step above, not just your existing val split -- that's the number that tells you whether this actually helped generalization.


## 29. [Improvement] Day 6 — Combined Final Evaluation

Brings together whichever combination of (best Phase-2 variant) + (checkpoint averaging) + (TTA) + (calibration) + (chosen operating threshold) performed best across Days 1-5, evaluated on ALL three sets: local val, clean ISIC subset, and the new multi-source held-out set from Day 5. Fill in MODEL_PATHS_TO_ENSEMBLE and re-run Sections 23-26 against each test set in turn by swapping `validation_generator` for the relevant generator, then tabulate all results here for the defense document.


In [ ]:

# Template for the final comparison table -- fill in after running Days 1-5 across all test sets.
final_comparison = [
    # (config_name,                    dataset,        recall, precision, f2, auc_pr)
    ("Baseline (BCE, block7, thresh=0.5)",       "Local val",      None, None, None, None),
    ("Baseline (BCE, block7, thresh=0.5)",       "Clean ISIC",     None, None, None, None),
    ("+ Best threshold",                          "Local val",      None, None, None, None),
    ("+ Best threshold",                          "Clean ISIC",     None, None, None, None),
    ("+ TTA",                                     "Local val",      None, None, None, None),
    ("+ TTA",                                     "Clean ISIC",     None, None, None, None),
    ("+ Checkpoint ensemble",                     "Local val",      None, None, None, None),
    ("+ Checkpoint ensemble",                     "Clean ISIC",     None, None, None, None),
    ("+ Calibration",                             "Local val",      None, None, None, None),
    ("+ Calibration",                             "Clean ISIC",     None, None, None, None),
    ("Best Phase-2 variant (fill in name)",       "Local val",      None, None, None, None),
    ("Best Phase-2 variant (fill in name)",       "Clean ISIC",     None, None, None, None),
    ("Best Phase-2 variant (fill in name)",       "New multi-source", None, None, None, None),
]

import pandas as pd
df_final = pd.DataFrame(final_comparison,
    columns=["Configuration", "Dataset", "Recall", "Precision", "F2", "AUC-PR"])
print(df_final.to_string(index=False))
print("\nFill in the None values as each Day 1-5 cell is run against each dataset,")
print("then this table drops straight into the defense document's results section.")
